# ArgLLM baseline (DF-QuAD) — GPT-OSS 120B on αNLI (500 claims, sample hash 24b904e78d)

Original ArgLLM prompts, depth-one tree, 0.5 Base and Est. Base.  
Run on Kaggle; the API key is read from Kaggle secrets (`OPENROUTER_API_KEY`). Outputs are kept so the reported numbers can be checked.

In [ ]:
# ===== DF-QuAD PART 1: setup + sample + original ArgLLM prompts =====
!pip install -q openai datasets scikit-learn

import os, re, json, time, pickle, hashlib, zipfile, io, urllib.request
import numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ---- model + dataset ----
MODEL         = "gptoss"
MODELS = {"gptoss": ("openai/gpt-oss-120b",   "GPT-OSS 120B"),
          "gemma":  ("google/gemma-4-31b-it", "Gemma 4 31B"),
          "llama":  ("meta-llama/llama-3.3-70b-instruct", "Llama 3.3 70B")}
MODEL_NAME, LABEL = MODELS[MODEL]
DATASET       = "aNLI"
EXPECTED_HASH = "24b904e78d"
RUN_NAME      = f"dfquad_{MODEL}_{DATASET.lower()}"
WORKERS       = 6

# ---- decoding parameters of the original study ----
TEMPERATURE, TOP_P = 0.7, 0.95
BASE_TOKENS = 4000 if "gpt-oss" in MODEL_NAME else 128     # reasoning model needs room

WORK_DIR = Path("/kaggle/working") / RUN_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT = WORK_DIR / f"{RUN_NAME}_ckpt.pkl"

client = OpenAI(api_key=UserSecretsClient().get_secret("OPENROUTER_API_KEY"),
                base_url="https://openrouter.ai/api/v1", timeout=240)

# ---- rebuild the same αNLI sample ----
def load_anli_dev():
    try:
        url = "https://storage.googleapis.com/ai2-mosaic/public/abductive-commonsense-reasoning-iclr2020/anli.zip"
        zf = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=120).read()))
        names = zf.namelist()
        jl = [n for n in names if n.endswith("dev.jsonl")][0]
        lb = [n for n in names if n.endswith("dev-labels.lst")][0]
        rows = [json.loads(l) for l in zf.read(jl).decode().splitlines() if l.strip()]
        df = pd.DataFrame(rows)
        df["label"] = [int(l) for l in zf.read(lb).decode().split()]
        print("loaded αNLI dev from the official zip")
        return df[["obs1", "obs2", "hyp1", "hyp2", "label"]]
    except Exception as e:
        print("official zip failed -> Hugging Face fallback:", str(e)[:100])
        from datasets import load_dataset
        ds = load_dataset("allenai/art", split="validation").to_pandas()
        ds = ds.rename(columns={"observation_1": "obs1", "observation_2": "obs2",
                                "hypothesis_1": "hyp1", "hypothesis_2": "hyp2"})
        return ds[["obs1", "obs2", "hyp1", "hyp2", "label"]]

def sentence(s):
    s = str(s).strip()
    return s if s.endswith((".", "!", "?")) else s + "."

raw = load_anli_dev()
raw = raw[raw["label"].isin([1, 2])].copy()
for c in ["obs1", "obs2", "hyp1", "hyp2"]:
    raw[c] = raw[c].astype(str).str.strip()
raw["item_id"] = [hashlib.md5("|".join(r).encode()).hexdigest()[:12]
                  for r in raw[["obs1", "obs2", "hyp1", "hyp2"]].values.tolist()]
raw = raw.drop_duplicates("item_id").sort_values("item_id").reset_index(drop=True)

SEED = 42
items = raw.sample(n=500, random_state=SEED).reset_index(drop=True)
gold = np.array([1] * 250 + [0] * 250)[np.random.RandomState(SEED).permutation(500)]

def make_claim(r, g):
    right = r["hyp1"] if r["label"] == 1 else r["hyp2"]
    wrong = r["hyp2"] if r["label"] == 1 else r["hyp1"]
    return f"{sentence(r['obs1'])} {sentence(r['obs2'])} This is because: {sentence(right if g == 1 else wrong)}"

claims = [make_claim(r, g) for (_, r), g in zip(items.iterrows(), gold)]
gold = gold.astype(int).tolist()
SAMPLE_HASH = hashlib.md5("|".join(f"{s}:{g}" for s, g in zip(items["item_id"], gold)).encode()).hexdigest()[:10]
print("claims:", len(claims), "| true:", sum(gold), "| SAMPLE HASH:", SAMPLE_HASH)
assert SAMPLE_HASH == EXPECTED_HASH, "Different sample!"

# ---- LLM call ----
def llm_raw(prompt, max_tokens=BASE_TOKENS):
    last = None
    for a in range(1, 7):
        try:
            r = client.chat.completions.create(
                model=MODEL_NAME, temperature=TEMPERATURE, top_p=TOP_P,
                max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}])
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            last = e
            if getattr(e, "status_code", None) == 402 or "Error code: 402" in str(e):
                raise RuntimeError("no API credit left")
            time.sleep(10 * a)
    raise RuntimeError(f"failed: {last}")

# ---- ORIGINAL ArgLLM prompts (prompt.py: ArgumentMiningPrompts.chatgpt) ----
def am_prompt(statement, support):
    return f"""
        Claim: {statement}

        Instructions:
            Provide a concise argument {"supporting" if support else "opposing"} the claim in less than 2 sentences.
            Utilize critical thinking and logical reasoning in your argument.
            Ensure clarity in your reasoning.
            Avoid circular reasoning or fallacious arguments.
            If you cannot return a valid and convincing argument for this claim, reply N/A.

        {"Supporting" if support else "Opposing"} Argument for '{statement}':"""

def am_format(argument):                      # original formatter
    if "N/A" in argument or "n/a" in argument:
        return "N/A"
    return re.sub(" +", " ", "".join(re.findall(r"(.*[.?!])",
                  argument.split("\n\n")[0])).strip('"\n'))

# ---- ORIGINAL ArgLLM prompts (prompt.py: UncertaintyEvaluatorPrompts.analyst, numeric) ----
def ue_prompt(statement, claim=None, support=False, topic=False):
    if topic:
        instructions = (
            "You are an analyst evaluating the validity of statements. "
            f'For the statement:\n\nStatement: "{statement}"\n\nplease give your confidence '
            f"that the statement is correct, accurate and truthful. "
            f"Your response should be between 0% and 100% with 0% indicating that the "
            f"considered statement is definitely invalid, 100% indicating that the considered statement is ")
    else:
        instructions = (
            "You are an analyst evaluating the validity and relevance of arguments. "
            f'For the argument:\n\nArgument: "{statement}"\n\nplease give your confidence '
            f"that the argument presents a compelling case {'in favour of' if support else 'against'} "
            f'the statement:\n\nStatement: "{claim}"\n\nYour assessment should be based '
            f"on how well the argument {'supports' if support else 'refutes'} the considered "
            "statement as well as the correctness, accuracy and truthfulness of the given argument. "
            f"Your response should be between 0% and 100% with 0% indicating that the "
            f"considered argument is definitely invalid, 100% indicating that the considered argument is ")
    return instructions + (
        "definitely valid and values in between indicating various levels of "
        "uncertainty. Your estimates should be well-calibrated, so feel free to "
        "err on the side of caution and output moderate probabilities if you are "
        "not completely sure in your assessment. "
        "Please respond in the following form:"
        "\n\nLikelihood: The predicted likelihood that the considered "
        f"{'argument' if not topic else 'statement'} is valid")

def read_likelihood(text):
    """API version of the forced 'Likelihood:' prefix: read the number from free text."""
    t = text or ""
    for pat in [r"Likelihood:\s*\**\s*(\d{1,3}(?:\.\d+)?)\s*%?", r"(\d{1,3}(?:\.\d+)?)\s*%"]:
        m = re.search(pat, t, flags=re.IGNORECASE)
        if m and 0 <= float(m.group(1)) <= 100:
            return float(m.group(1)) / 100
    return None

print("Part 1 ready |", LABEL, "|", DATASET, "| tokens", BASE_TOKENS,
      "| temperature", TEMPERATURE, "| top_p", TOP_P)
t = llm_raw(am_prompt("Paris is the capital of France.", True))
print("TEST argument:", repr(am_format(t))[:200])

loaded αNLI dev from the official zip
claims: 500 | true: 250 | SAMPLE HASH: 24b904e78d
Part 1 ready | GPT-OSS 120B | aNLI | tokens 4000 | temperature 0.7 | top_p 0.95
TEST argument: 'Paris is designated as France’s capital by its constitution and law, and it houses the primary institutions of national governance—including the President’s office, the Parliament, and the Supreme Co


In [ ]:
# ===== DF-QuAD PART 2: run + DF-QuAD + report + backup =====
def base_score(statement, claim=None, support=False, topic=False):
    """Original rule: an N/A argument gets 0.0. If no number can be read,
    ask again with a longer answer, up to three times; then default 0.5."""
    if statement == "N/A":
        return 0.0, False
    p = ue_prompt(statement, claim=claim, support=support, topic=topic)
    for k in range(3):
        v = read_likelihood(llm_raw(p, max_tokens=BASE_TOKENS * (2 ** k)))
        if v is not None:
            return v, False
    return 0.5, True                               # unreadable -> default 0.5

def dfquad_manual(w, v_sup, v_att):
    """DF-QuAD for a claim with one supporter and one attacker (depth 1),
    same fixed point as Uncertainpy's ContinuousDFQuADModel."""
    g = v_sup - v_att
    return w + (1 - w) * g if g > 0 else w + w * g

def run_claim(i):
    c = claims[i]
    sup = am_format(llm_raw(am_prompt(c, True)))
    att = am_format(llm_raw(am_prompt(c, False)))
    s_sup, u1 = base_score(sup, claim=c, support=True)
    s_att, u2 = base_score(att, claim=c, support=False)
    s_top, u3 = base_score(c, topic=True)
    return {"i": i, "claim": c, "gold": gold[i],
            "support": sup, "attack": att,
            "w_support": s_sup, "w_attack": s_att, "w_claim_est": s_top,
            "unreadable": int(u1) + int(u2) + int(u3),
            "strength_05": dfquad_manual(0.5, s_sup, s_att),
            "strength_est": dfquad_manual(s_top, s_sup, s_att)}

def load_ck():
    return pickle.load(open(CKPT, "rb")) if CKPT.exists() else {}

def save_ck(d):
    tmp = CKPT.with_suffix(".tmp"); pickle.dump(d, open(tmp, "wb")); tmp.replace(CKPT)

def run_pass(label):
    d = load_ck()
    todo = [i for i in range(len(claims)) if i not in d]
    print(f"[{label}] have {len(d)}/{len(claims)}, running {len(todo)}", flush=True)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futs = {pool.submit(run_claim, i): i for i in todo}
        for f in as_completed(futs):
            try:
                v = f.result()
            except Exception as e:
                if "no API credit" in str(e):
                    raise
                print(f"  error at {futs[f]}: {str(e)[:100]}", flush=True)
                continue
            d[v["i"]] = v
            save_ck(d)
            if len(d) % 50 == 0:
                print(f"  {len(d)}/{len(claims)} | {(time.time() - t0) / 60:.0f} min", flush=True)
    return load_ck()

d = run_pass("pass 1")
for k in range(2):
    if len(d) < len(claims):
        time.sleep(30)
        d = run_pass(f"pass {k + 2}")
    else:
        print(f"[pass {k + 2}] have {len(d)}/{len(claims)}, running 0")
        break

# ---- report ----
rows = [d[k] for k in sorted(d)]
n = len(rows)
g = [r["gold"] for r in rows]

def metrics(name, pred):
    tn, fp, fn, tp = confusion_matrix(g, pred, labels=[0, 1]).ravel()
    return {"method": name, "n": n,
            "acc": round(accuracy_score(g, pred) * 100, 1),
            "prec": round(precision_score(g, pred, zero_division=0) * 100, 1),
            "rec": round(recall_score(g, pred, zero_division=0) * 100, 1),
            "f1": round(f1_score(g, pred, zero_division=0) * 100, 1),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}

table = pd.DataFrame([
    metrics("DF-QuAD 0.5 Base",  [int(r["strength_05"] > 0.5) for r in rows]),
    metrics("DF-QuAD Est. Base", [int(r["strength_est"] > 0.5) for r in rows])])
pd.set_option("display.width", 220)
print(f"\n=== FINAL: {LABEL} | {DATASET} | DF-QuAD (no facts) | sample {SAMPLE_HASH} ({n}/{len(claims)}) ===")
print(table.to_string(index=False))
print(f"N/A sup: {sum(r['support'] == 'N/A' for r in rows)} | N/A att: {sum(r['attack'] == 'N/A' for r in rows)}")
print(f"unreadable strength scores (default 0.5): {sum(r['unreadable'] for r in rows)}")

# ---- save + green Download button ----
import base64
from IPython.display import HTML, display
pd.DataFrame(rows).to_csv(WORK_DIR / f"{RUN_NAME}_per_claim.csv", index=False)
table.to_csv(WORK_DIR / f"{RUN_NAME}_results.csv", index=False)
zp = WORK_DIR / f"{RUN_NAME}_{n}_backup.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [CKPT, WORK_DIR / f"{RUN_NAME}_per_claim.csv", WORK_DIR / f"{RUN_NAME}_results.csv"]:
        z.write(f, arcname=f.name)
b64 = base64.b64encode(open(zp, "rb").read()).decode()
display(HTML(
    f'<a download="{zp.name}" href="data:application/zip;base64,{b64}" '
    f'style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;'
    f'border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download {zp.name}</a>'))

[pass 1] have 0/500, running 500
  50/500 | 4 min
  100/500 | 9 min
  150/500 | 12 min
  200/500 | 16 min
  250/500 | 19 min
  300/500 | 23 min
  350/500 | 26 min
  400/500 | 30 min
  450/500 | 33 min
  500/500 | 39 min
[pass 2] have 500/500, running 0

=== FINAL: GPT-OSS 120B | aNLI | DF-QuAD (no facts) | sample 24b904e78d (500/500) ===
           method   n  acc  prec  rec   f1  TN  FP  FN  TP
 DF-QuAD 0.5 Base 500 53.8  55.4 38.8 45.6 172  78 153  97
DF-QuAD Est. Base 500 55.2  65.9 21.6 32.5 222  28 196  54
N/A sup: 39 | N/A att: 0
unreadable strength scores (default 0.5): 0
